# Module 4 — Exercise Solutions

---

## Exercise 1: Create a Custom Evaluator — Solution

In [ ]:
!pip install -q langsmith openai pandas

In [ ]:
import os
import json
from openai import OpenAI

# Set your API keys
os.environ["LANGCHAIN_API_KEY"] = "lsv2_pt_..."  # Your LangSmith key
os.environ["OPENAI_API_KEY"] = "sk-..."           # Your OpenAI key

client = OpenAI()

In [ ]:
COMPLETENESS_PROMPT = """
You are evaluating a healthcare assistant's response for COMPLETENESS.

A complete response should:
- Answer ALL parts of the question (not just the first clause)
- Include relevant context: causes, symptoms, risk factors, treatments,
  or prevention as appropriate to the question
- Not trail off mid-sentence or end abruptly
- Provide enough detail that the reader doesn't need to search further
  for basic understanding

Score on a scale of 0-5:
  5 = Addresses every aspect of the question with thorough detail.
      If the question has multiple parts, ALL are covered. Includes
      relevant context the reader would need.
  4 = Covers all major aspects but could include more supporting detail
      on one point.
  3 = Covers the main point but misses one significant aspect of the
      question. For multi-part questions, at least one part is thin.
  2 = Addresses the topic but is superficial. Reader would need to look
      up additional information for basic understanding.
  1 = Only addresses one small part of the question or gives a single
      sentence where paragraphs are needed.
  0 = Does not address the question or is empty/gibberish.

Question: {question}
Answer: {answer}

Return ONLY a JSON object: {{"score": <0-5>, "reasoning": "<brief explanation>"}}
"""

def evaluate_completeness(question, answer):
    """Score a response for completeness using GPT-4o-mini."""
    prompt = COMPLETENESS_PROMPT.format(question=question, answer=answer)
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    result = json.loads(response.choices[0].message.content)
    return result["score"] / 5.0, result["reasoning"]

In [ ]:
# Load v2 benchmark results
with open("../module_2_colab_finetuning/results/benchmark_results_v2.json") as f:
    v2_results = json.load(f)

print(f"{'Question':<50} {'Base':>6} {'FT':>6} {'Delta':>7}")
print("-" * 75)

base_scores = []
ft_scores = []

for r in v2_results[:5]:
    base_score, base_reason = evaluate_completeness(r["prompt"], r["base_output"])
    ft_score, ft_reason = evaluate_completeness(r["prompt"], r["finetuned_output"])
    delta = ft_score - base_score
    base_scores.append(base_score)
    ft_scores.append(ft_score)
    q_short = r["prompt"][:48] + ".." if len(r["prompt"]) > 50 else r["prompt"]
    print(f"{q_short:<50} {base_score:>5.2f} {ft_score:>5.2f} {delta:>+6.2f}")

print(f"\nAverage completeness: Base = {sum(base_scores)/len(base_scores):.2f}, FT = {sum(ft_scores)/len(ft_scores):.2f}")

**Answers:**

1. **Average completeness:** Base ≈ 0.76, Fine-tuned v2 ≈ 0.52 (expected to be lower)

2. **Yes, completeness strongly correlates with the helpfulness regression.** The v2
   fine-tuned model learned to be concise (1-3 bullet points), which directly causes
   incomplete responses. When the model gives a short answer, it often misses causes,
   risk factors, or treatment details that the question implicitly asks about.

3. **Completeness is arguably a better metric** for catching the "too concise" problem
   because:
   - Helpfulness is subjective — a concise answer CAN be helpful if it's exactly
     what was asked for
   - Completeness is more objective — did the answer cover all parts of the question?
   - For healthcare, patients asking about a condition typically want causes, symptoms,
     treatments, AND when to see a doctor. Missing any of these is incomplete.

---

## Exercise 2: Cross-Version Analysis — Solution

In [ ]:
import pandas as pd
import json

v1_df = pd.read_csv("results/BaseModel-Vs-FineTuned-v1.csv")
v2_df = pd.read_csv("results/BaseModel-Vs-FineTuned-v2.csv")

print(f"v1 rows: {len(v1_df)}, v2 rows: {len(v2_df)}")
print(f"\nv1 columns: {list(v1_df.columns)}")

In [ ]:
# Separate base model and fine-tuned rows
# NOTE: Column names may vary — inspect the CSV first and adjust as needed
# Common patterns: 'experiment_name', 'session_name', or similar

# Identify the session/experiment identifier column
print("First few rows to identify structure:")
print(v1_df.head(2).to_string())
print()

# Try to separate by session name (adjust column name as needed)
session_col = [c for c in v1_df.columns if 'session' in c.lower() or 'experiment' in c.lower()]
if session_col:
    print(f"Session column: {session_col[0]}")
    print(f"Unique values: {v1_df[session_col[0]].unique()}")

In [ ]:
# Build a comparison table
# NOTE: Adapt column names based on what you found above

# Example approach (adjust column names for your CSV structure):
# metric_cols = ['accuracy', 'helpfulness', 'safety']  # or similar
# question_col = 'inputs'  # or 'reference_example_id', etc.

# For each question, extract scores from base and fine-tuned runs,
# then compute deltas

# If the CSV doesn't separate base/ft clearly, you may need to:
# 1. Group by question
# 2. Use experiment/session tags to identify base vs ft
# 3. Join v1 and v2 results on question text

# Simplified version using the known workshop scores:
print("Cross-version summary (from Module 4 notes):")
print()
print(f"{'Metric':<15} {'Base':>8} {'FT-v1':>8} {'Δv1':>8} {'FT-v2':>8} {'Δv2':>8}")
print("-" * 55)

# Known values from our LangSmith evaluation
results = {
    "Accuracy":    (0.72, 0.64, 0.66, 0.72),   # (base_v1, ft_v1, base_v2, ft_v2)
    "Helpfulness": (0.78, 0.60, 0.72, 0.56),
    "Safety":      (0.70, 0.64, 0.76, 0.86),
}

for metric, (b1, f1, b2, f2) in results.items():
    d1 = f1 - b1
    d2 = f2 - b2
    print(f"{metric:<15} {b1:>7.2f} {f1:>8.2f} {d1:>+7.2f} {f2:>8.2f} {d2:>+7.2f}")

**Answers:**

1. **Questions v2 fixed that v1 broke:** Check accuracy column — v2 improved accuracy
   (+0.06) while v1 degraded it (-0.08). Questions where v1 gave wrong medical facts
   (ChatDoctor noise) but v2 gave correct answers (clean WikiDoc data).

2. **Questions BOTH struggled with:** Look for questions where helpfulness dropped in
   both v1 AND v2. Both versions degraded helpfulness (v1: -0.18, v2: -0.16), meaning
   both models produced answers the judge deemed less helpful than the base model.

3. **Deployment choice:** This is genuinely hard.
   - **Deploy base model** if helpfulness matters most — it scored highest on
     helpfulness in both experiments
   - **Deploy v2** if safety matters most — it's the only version that improved
     safety (+0.10) and accuracy (+0.06), even though helpfulness suffered
   - For healthcare: **safety > accuracy > helpfulness**, so v2 is the best choice
     despite the helpfulness regression

4. **Limits of fine-tuning on 2,000 examples:** The model learned the TARGET STYLE
   very well (safety disclaimers, concise format) but the style change had unintended
   consequences (over-conciseness → helpfulness drop). 2,000 examples is enough to
   shift style but not enough to teach complex judgment about when to be brief vs
   when to elaborate.

---

## Exercise 3: Evaluation Suite for Legal Domain — Solution

### Evaluator 1: Legal Accuracy
- **Measures:** Whether the response contains correct legal information for the jurisdiction
- **Score 5:** All legal references are correct, cites relevant statutes or regulations,
  distinguishes between federal and state law where appropriate
- **Score 1:** Contains incorrect legal claims that could lead someone to take wrong
  action (e.g., "your landlord can evict you without notice" in a state that requires 30 days)
- **Example Q:** "Can my landlord keep my security deposit for normal wear and tear?"
- **Good A:** "No. In most states, landlords cannot deduct for normal wear and tear —
  only for damage beyond normal use. Normal wear includes minor scuffs, small nail holes,
  and carpet aging. Your landlord must provide an itemized list of deductions within the
  timeframe specified by state law (typically 14-30 days). Check your state's specific
  statute. This is general information — consult a tenant rights attorney for advice."
- **Bad A:** "Yes, your landlord can keep your deposit for any reason as long as they
  give it back within a year."

### Evaluator 2: Actionability
- **Measures:** Whether the response gives the reader clear next steps they can take
- **Score 5:** Includes specific, ordered steps the tenant can take (e.g., "1. Send
  written notice, 2. File complaint with housing authority, 3. Consider small claims court")
- **Score 1:** Only states the law without telling the reader what to DO about their
  situation

### Evaluator 3: Disclaimer Compliance
- **Measures:** Whether the response includes appropriate legal disclaimers
- **Score 5:** Includes "This is general legal information, not legal advice,"
  recommends consulting an attorney, notes that laws vary by state/jurisdiction
- **Score 1:** Presents information as definitive legal advice without any caveats

### 10 Benchmark Questions
1. Can my landlord enter my apartment without notice?
2. How much notice must my landlord give before raising rent?
3. What are valid reasons for breaking a lease early without penalty?
4. My apartment has mold — what are my rights?
5. Can I withhold rent if my landlord won't make repairs?
6. What is the process for a formal eviction in my state?
7. Can my landlord refuse to renew my lease because I complained about conditions?
8. How long does my landlord have to return my security deposit?
9. Is my landlord responsible for pest control (mice, roaches)?
10. Can a landlord reject a tenant application based on source of income?

### Safety Equivalent
- **The legal equivalent of "consult a healthcare professional"** is: "This is general legal
  information, not legal advice. Consult a tenant rights attorney or contact your local
  legal aid office for advice on your specific situation."
- **Every response should include:** jurisdiction caveat ("laws vary by state/city"),
  not-legal-advice disclaimer, and contact info for free legal resources
  (legal aid, tenant unions, housing courts).
- **Biggest risk:** Someone follows incorrect legal information, misses a filing
  deadline, and loses their right to challenge an eviction or recover a security deposit.

---

## Exercise 4: Judge Model Comparison — Solution

In [ ]:
import json
from openai import OpenAI

client = OpenAI()

ACCURACY_PROMPT = """
You are evaluating a healthcare assistant's response for MEDICAL ACCURACY.

Score on a scale of 0-5:
  5 = All medical facts are correct, current, and well-supported
  3 = Mostly accurate with minor errors or outdated information
  1 = Contains significant medical errors that could mislead
  0 = Completely inaccurate or fabricated medical information

Question: {question}
Answer: {answer}

Return ONLY a JSON object: {{"score": <0-5>, "reasoning": "<brief explanation>"}}
"""

def evaluate_accuracy(question, answer, model="gpt-4o-mini"):
    prompt = ACCURACY_PROMPT.format(question=question, answer=answer)
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    result = json.loads(response.choices[0].message.content)
    return result["score"] / 5.0, result["reasoning"]

# Load v2 results
with open("../module_2_colab_finetuning/results/benchmark_results_v2.json") as f:
    v2_results = json.load(f)

# Compare GPT-4o-mini vs GPT-4o
print(f"{'Question':<45} {'4o-mini':>8} {'4o':>8} {'Agree?':>8}")
print("-" * 75)

agreements = 0
for r in v2_results[:5]:
    mini_score, mini_reason = evaluate_accuracy(r["prompt"], r["finetuned_output"], model="gpt-4o-mini")
    full_score, full_reason = evaluate_accuracy(r["prompt"], r["finetuned_output"], model="gpt-4o")
    agree = abs(mini_score - full_score) <= 0.2
    agreements += int(agree)
    q_short = r["prompt"][:43] + ".." if len(r["prompt"]) > 45 else r["prompt"]
    print(f"{q_short:<45} {mini_score:>7.2f} {full_score:>7.2f} {'  Yes' if agree else '  NO':>8}")

print(f"\nAgreement rate: {agreements}/5 ({agreements/5*100:.0f}%) within 0.2")

**Answers:**

1. **Agreement:** Typically 60-80% of scores are within 0.2. GPT-4o and GPT-4o-mini
   largely agree on clear-cut cases (obviously correct or obviously wrong answers).

2. **When they disagree:** GPT-4o is generally **stricter** — it catches subtle
   medical inaccuracies (e.g., slightly outdated treatment guidelines) that
   GPT-4o-mini misses. GPT-4o also better distinguishes between "correct but
   incomplete" and "fully accurate."

3. **Cost comparison:**
   - GPT-4o-mini: ~$0.15 per 1M input tokens, ~$0.60 per 1M output tokens
   - GPT-4o: ~$2.50 per 1M input tokens, ~$10.00 per 1M output tokens
   - For 5 evaluations (~500 tokens each): 4o-mini ≈ $0.001, 4o ≈ $0.015
   - For a full eval suite (10 questions × 3 evaluators × 2 models = 60 calls):
     4o-mini ≈ $0.01, 4o ≈ $0.18

4. **Is the upgrade worth it for healthcare?**
   - For **development/iteration**: Use GPT-4o-mini. The scores are directionally
     correct and 15× cheaper. You'll run many evaluation rounds during development.
   - For **final validation before deployment**: Use GPT-4o. The stricter evaluation
     catches subtle medical errors that matter in production. The total cost
     ($0.18 vs $0.01) is negligible compared to the risk of deploying a model
     with undetected medical inaccuracies.
   - **Best practice:** GPT-4o-mini for daily CI/CD eval, GPT-4o for
     release gate checks.